# ProductIQ Phase 2 — Step 2.3: Data Quality & Validation

This notebook loads the raw AJIO dataset through the ingestion layer and runs the validation framework.

Validation inspects data quality rules without cleaning, normalization, or mutation.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path

import pandas as pd

from data.ingestion import AjioDataLoader
from data.validation import AjioDataValidator
from data.validation.ajio import _parse_price_for_validation


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

loader = AjioDataLoader()
validator = AjioDataValidator()
dataframe = loader.load(DATASET_PATH)

In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Shape: {dataframe.shape}")

In [ ]:
before_validation = deepcopy(dataframe)
report = validator.validate(dataframe)

print(report.summary())
print("\nHas failures:", report.has_failures)
print("Has warnings:", report.has_warnings)

In [ ]:
price_anomalies = dataframe[
    dataframe.apply(
        lambda row: _parse_price_for_validation(row["Discount Price (in Rs.)"])
        > _parse_price_for_validation(row["Original Price (in Rs.)"]),
        axis=1,
    )
][[
    "Id_Product",
    "Brand",
    "Discount Price (in Rs.)",
    "Original Price (in Rs.)",
]]

print(f"Price anomalies detected: {len(price_anomalies)}")
price_anomalies

In [ ]:
pd.testing.assert_frame_equal(dataframe, before_validation)
print("Validation did not mutate the DataFrame.")

In [ ]:
price_findings = report.findings_by_category("PRICE BUSINESS RULES")
for finding in price_findings:
    print(finding)